# Generate the unanchored Velocileptors PT training tables

This notebook builds the first broad training set for the factorized cosmological $P(k)$ emulator.

**Design:** 300 cosmologies sampled across a revised, fixed 8D cosmological domain with a scrambled Latin hypercube. The dark-energy bounds are chosen to avoid the extreme early-dark-energy combinations encountered in the first broad design. For each cosmology, save the native Velocileptors PT component table. The table contains the cosmology-dependent information; nuisance dependence is applied later through the exact analytic 19-component contraction.

This is intentionally **not** the 52-anchor production comparison. We are first building a working emulator dataset with broad coverage; the training-set size can be reduced after the pipeline and emulator are validated.

**Expected table:** $(n_\ell,n_{k,\mathrm{native}},19)=(3,117,19)$, with $\ell=[0,2,4]$. The native PT $k$ grid is saved alongside every table; do not replace it with the 36-point output grid.

Run from the project checkout in the configured NERSC Jupyter environment. The project package is imported as `src`; this notebook does not modify `sys.path`.

In [ ]:
from pathlib import Path
import json
import os
import time
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import qmc

np.set_printoptions(precision=7, suppress=True)
print("Python:", platform.python_version())

## 1. Project inputs and imports

Import the project cosmology and theory interfaces. The LHS domain is specified explicitly below; this notebook no longer needs the Abacus cosmology catalogue to construct the sampling bounds.

In [ ]:
from src.cosmology import COSMOLOGY_PARAMETERS
from src.theory import (
    TheoryManager,
    Z_FID,
    NBAR_FID,
    ELL_FID,
    K_FID,
)

EXPECTED_COSMOLOGY_PARAMETERS = [
    "omega_b", "omega_cdm", "sigma8_cb", "n_s",
    "alpha_s", "N_ur", "w0_fld", "wa_fld",
]
assert list(COSMOLOGY_PARAMETERS) == EXPECTED_COSMOLOGY_PARAMETERS
assert np.isclose(Z_FID, 0.8)

print("Redshift:", Z_FID)
print("Output multipoles:", ELL_FID)
print("Theory output k grid:", len(K_FID), (float(K_FID.min()), float(K_FID.max())))
print("Cosmology coordinates:", COSMOLOGY_PARAMETERS)

## 2. Define the revised fixed 8D cosmological domain

Use explicit bounds rather than expanding every Abacus coordinate by a common factor. The six non-dark-energy parameters retain broad coverage. The $w_0$ and $w_a$ limits are tightened to avoid the extreme early-time dark-energy behavior that caused the fixed-acoustic-scale conversion to fail for some points in the previous design.

The selected limits guarantee $w_0+w_a\leq -0.20$ throughout the rectangular domain. This is a practical training-domain choice, not a claim that the boundary is a fundamental physical limit.

In [ ]:
# Fixed bounds for the revised 8D training design.
expanded_bounds = {
    "omega_b":    (0.0205, 0.0245),
    "omega_cdm":  (0.1010, 0.1420),
    "sigma8_cb":  (0.6650, 0.9550),
    "n_s":        (0.8950, 1.0320),
    "alpha_s":    (-0.0420, 0.0420),
    "N_ur":       (1.0800, 3.0000),
    "w0_fld":     (-1.3000, -0.8000),
    "wa_fld":     (-0.7000, 0.6000),
}

if set(expanded_bounds) != set(COSMOLOGY_PARAMETERS):
    raise ValueError("The fixed bounds must specify every cosmology parameter exactly once.")

bound_records = [
    {
        "parameter": name,
        "minimum": float(expanded_bounds[name][0]),
        "maximum": float(expanded_bounds[name][1]),
    }
    for name in COSMOLOGY_PARAMETERS
]
bounds_df = pd.DataFrame(bound_records).set_index("parameter")
display(bounds_df)

# The maximum possible w0 + wa occurs at the upper bound of both parameters.
max_early_time_w = expanded_bounds["w0_fld"][1] + expanded_bounds["wa_fld"][1]
assert max_early_time_w <= -0.20
print(f"Maximum w0 + wa across the domain: {max_early_time_w:.3f}")

## 3. Generate the 300-cosmology Latin hypercube

The cosmological design is an independently generated, scrambled 8D Latin hypercube over the fixed bounds above. The seed and exact design are saved for reproducibility. No nuisance parameters are sampled here: each cosmology contributes one cosmology-dependent PT component table, which is sufficient for the factorized emulator.

In [ ]:
N_COSMOLOGIES = 300
COSMOLOGY_SEED = 20261110

if N_COSMOLOGIES < 1:
    raise ValueError("N_COSMOLOGIES must be positive.")

sampler = qmc.LatinHypercube(
    d=len(COSMOLOGY_PARAMETERS),
    scramble=True,
    seed=COSMOLOGY_SEED,
)
unit_design = sampler.random(n=N_COSMOLOGIES)

lower = np.array([expanded_bounds[name][0] for name in COSMOLOGY_PARAMETERS])
upper = np.array([expanded_bounds[name][1] for name in COSMOLOGY_PARAMETERS])
physical_design = qmc.scale(unit_design, lower, upper)

cosmology_design = pd.DataFrame(
    physical_design,
    columns=COSMOLOGY_PARAMETERS,
)
cosmology_design.insert(
    0, "cosmology_index", np.arange(N_COSMOLOGIES, dtype=int)
)

if cosmology_design[COSMOLOGY_PARAMETERS].isna().any().any():
    raise ValueError("Non-finite values in cosmology design.")
if not np.isfinite(cosmology_design[COSMOLOGY_PARAMETERS].to_numpy()).all():
    raise ValueError("Non-finite values in cosmology design.")

# Confirm the sampled dark-energy combinations obey the intended bound.
if (cosmology_design["w0_fld"] + cosmology_design["wa_fld"]).max() > -0.20:
    raise ValueError("Sample contains an unintended early-dark-energy combination.")

display(cosmology_design.head())
print(f"Generated {len(cosmology_design)} cosmologies with LHS seed {COSMOLOGY_SEED}.")

## 4. Configure the PT-table calculation

The existing `TheoryManager` performs the required cosmology conversion (solving for $h$ and $A_s$ at fixed acoustic scale and target $\sigma_{8,cb}$), initializes the $z=0.8$ LRG-like Velocileptors theory, and provides a cache around the theory object.

We evaluate one harmless fiducial nuisance point to force the theory pipeline to initialize for each cosmology. The stored `pt.table` is the cosmology-dependent component table; the nuisance values used to trigger initialization are **not** part of the learned cosmological input.

In [ ]:
BASELINE_NUISANCE = {
    "bias": {"b1": 2.0, "b2": 0.0, "bs": 0.0, "b3": 0.0},
    "eft": {"alpha0": 0.0, "alpha2": 0.0, "alpha4": 0.0, "alpha6": 0.0},
    "stochastic": {"sn0": 0.0, "sn2": 0.0, "sn4": 0.0},
}

# One cached theory at a time limits memory use during a 250-cosmology run.
manager = TheoryManager(max_cache_size=1, z=Z_FID)

EXPECTED_N_ELL = 3
EXPECTED_N_COMPONENTS = 19
EXPECTED_N_K_NATIVE = 117

# Keep the table axis explicitly indexed. The physical meaning and ordering
# of the 19 entries are defined by the installed Velocileptors implementation
# and the coefficient-contraction helper, not guessed from their number.
COMPONENT_LABELS = [f"component_{i:02d}" for i in range(EXPECTED_N_COMPONENTS)]
assert len(COMPONENT_LABELS) == EXPECTED_N_COMPONENTS
print("Theory manager initialized.")
print("Fixed theory settings: z =", Z_FID, ", nbar =", NBAR_FID)

## 5. Identify and validate the native PT $k$ coordinate

The native $k$ grid is read from the initialized PT object. We deliberately do not assume it is the same as the 36-bin requested output grid. The PT component table and its native $k$ coordinate are saved together for every cosmology.

In [ ]:
def get_native_k_coordinate(pt, n_k):
    """Find a one-dimensional native-k coordinate matching the PT table."""
    candidates = []
    for name in dir(pt):
        if name.startswith("_") or "k" not in name.lower():
            continue
        try:
            value = getattr(pt, name)
            array = np.asarray(value)
        except Exception:
            continue
        if (
            array.ndim == 1
            and array.size == n_k
            and np.issubdtype(array.dtype, np.number)
            and np.isfinite(array).all()
        ):
            candidates.append((name, array.astype(float)))

    if not candidates:
        raise RuntimeError(
            "Could not identify the native PT k coordinate. "
            "Inspect the installed Velocileptors PT object before proceeding."
        )

    # Prefer the exact attribute name 'k'; otherwise use the first matching k-like attribute.
    for name, values in candidates:
        if name.lower() == "k":
            selected = (name, values)
            break
    else:
        selected = candidates[0]

    name, values = selected
    if not np.all(np.diff(values) > 0):
        raise ValueError(f"pt.{name} is not strictly increasing.")
    return name, values

print("Native-k helper defined.")

## 6. Generate and save one PT table per cosmology

Each output file contains:
- `pt_table`: the native cosmology-dependent component table, expected shape `(3, 117, 19)`;
- `k_native`: the corresponding native $k$ coordinate;
- `ells` and the ordered component labels;
- the exact eight cosmological input parameters;
- JSON metadata describing the cosmology index, domain, design seed, redshift, theory settings, and table layout.

Files are written atomically. The progress CSV is refreshed throughout the run, so completed cosmologies remain documented if a later calculation fails. Existing files are never silently overwritten.

In [ ]:
OUTPUT_ROOT = Path("./training_data")
RUN_NAME = "velocileptors_pt_lhs_v2_restricted"
RUN_DIR = OUTPUT_ROOT / RUN_NAME
TABLE_DIR = RUN_DIR / "pt_tables"

# Refuse to mix this run with a previous partial or complete run.
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise FileExistsError(
        f"{RUN_DIR} already exists and is not empty. "
        "Choose a new RUN_NAME or move the previous run before restarting."
    )

TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Save the exact cosmology design and run configuration before expensive theory calls.
cosmology_design.to_csv(RUN_DIR / "cosmology_design.csv", index=False)

run_config = {
    "purpose": "unanchored cosmological Velocileptors PT component training tables",
    "n_cosmologies": int(N_COSMOLOGIES),
    "cosmology_seed": int(COSMOLOGY_SEED),
    "sampling": "scrambled 8D Latin hypercube",
    "cosmology_parameters": list(COSMOLOGY_PARAMETERS),
    "domain_definition": "fixed revised 8D bounds; max(w0_fld + wa_fld) = -0.20",
    "expanded_bounds": {
        name: [float(bounds[0]), float(bounds[1])]
        for name, bounds in expanded_bounds.items()
    },
    "redshift": float(Z_FID),
    "ells": list(map(int, ELL_FID)),
    "nbar": float(NBAR_FID),
    "expected_pt_table_shape": [
        EXPECTED_N_ELL, EXPECTED_N_K_NATIVE, EXPECTED_N_COMPONENTS
    ],
    "component_labels": COMPONENT_LABELS,
    "baseline_nuisance_for_initialization_only": BASELINE_NUISANCE,
    "output_directory": str(RUN_DIR),
}
with open(RUN_DIR / "run_config.json", "w") as stream:
    json.dump(run_config, stream, indent=2)

progress_records = []
generation_start = time.perf_counter()
native_k_name = None
native_k_reference = None

for row in cosmology_design.itertuples(index=False):
    cosmology_index = int(row.cosmology_index)
    cosmo_params = {
        name: float(getattr(row, name))
        for name in COSMOLOGY_PARAMETERS
    }

    # Initialize and evaluate the theory at this cosmology.
    _ = manager.evaluate(cosmo_params, **BASELINE_NUISANCE)
    theory, _pipeline = manager.get_theory(cosmo_params)

    pt = getattr(theory, "pt", None)
    if pt is None or not hasattr(pt, "table"):
        raise RuntimeError(
            f"Cosmology {cosmology_index}: initialized theory has no pt.table."
        )

    pt_table = np.asarray(pt.table, dtype=float)
    if pt_table.ndim != 3:
        raise ValueError(
            f"Cosmology {cosmology_index}: expected a rank-3 PT table, "
            f"got shape {pt_table.shape}."
        )
    if pt_table.shape[0] != EXPECTED_N_ELL:
        raise ValueError(
            f"Cosmology {cosmology_index}: unexpected multipole axis {pt_table.shape}."
        )
    if pt_table.shape[-1] != EXPECTED_N_COMPONENTS:
        raise ValueError(
            f"Cosmology {cosmology_index}: expected 19 components, "
            f"got shape {pt_table.shape}."
        )
    if pt_table.shape[1] != EXPECTED_N_K_NATIVE:
        raise ValueError(
            f"Cosmology {cosmology_index}: expected {EXPECTED_N_K_NATIVE} native k bins, "
            f"got shape {pt_table.shape}."
        )
    if not np.isfinite(pt_table).all():
        raise ValueError(
            f"Cosmology {cosmology_index}: non-finite values in pt.table."
        )

    this_k_name, k_native = get_native_k_coordinate(pt, pt_table.shape[1])
    if native_k_reference is None:
        native_k_name = this_k_name
        native_k_reference = k_native.copy()
    elif not np.allclose(k_native, native_k_reference, rtol=0.0, atol=1e-12):
        raise ValueError(
            f"Native PT k grid changed at cosmology {cosmology_index}."
        )

    metadata = {
        "cosmology_index": cosmology_index,
        "cosmology": cosmo_params,
        "cosmology_seed": int(COSMOLOGY_SEED),
        "redshift": float(Z_FID),
        "ells": list(map(int, ELL_FID)),
        "native_k_attribute": native_k_name,
        "pt_table_shape": list(pt_table.shape),
        "component_labels": COMPONENT_LABELS,
        "nbar": float(NBAR_FID),
        "expanded_bounds": run_config["expanded_bounds"],
    }

    output_file = TABLE_DIR / f"cosmo_{cosmology_index:04d}.npz"
    temporary_file = output_file.with_suffix(".npz.tmp")
    with open(temporary_file, "wb") as stream:
        np.savez_compressed(
            stream,
            pt_table=pt_table,
            k_native=k_native,
            ells=np.asarray(ELL_FID, dtype=int),
            component_labels=np.asarray(COMPONENT_LABELS, dtype=str),
            cosmology_json=np.asarray(json.dumps(cosmo_params)),
            metadata_json=np.asarray(json.dumps(metadata)),
        )
    os.replace(temporary_file, output_file)

    elapsed = time.perf_counter() - generation_start
    progress_records.append({
        "cosmology_index": cosmology_index,
        "file": str(output_file),
        "table_shape": str(tuple(pt_table.shape)),
        "table_rms": float(np.sqrt(np.mean(pt_table**2))),
        "elapsed_seconds": float(elapsed),
    })

    # Persist progress after every completed cosmology.
    pd.DataFrame(progress_records).to_csv(
        RUN_DIR / "progress.csv", index=False
    )

    print(
        f"[{len(progress_records):3d}/{N_COSMOLOGIES}] "
        f"cosmology {cosmology_index:04d} saved; "
        f"elapsed {elapsed/60:.1f} min; "
        f"cache {manager.cache_info()}"
    )

generation_seconds = time.perf_counter() - generation_start
print("\nGeneration complete.")
print(f"Tables saved: {len(progress_records)}")
print(f"Total elapsed: {generation_seconds/60:.2f} minutes")
print("Native k attribute:", native_k_name)
print("Native k range:", (float(native_k_reference.min()), float(native_k_reference.max())))
print("Output directory:", RUN_DIR.resolve())

## 7. Validate the saved training tables

Reload every file without pickle support. Check the expected dimensions, finite values, identical native $k$ grids, unique cosmology indices, and consistency with the original design. These checks validate serialization and bookkeeping; the separate theory-reconstruction notebook validates the nuisance contraction.

In [ ]:
saved_files = sorted(TABLE_DIR.glob("cosmo_*.npz"))
if len(saved_files) != N_COSMOLOGIES:
    raise RuntimeError(
        f"Expected {N_COSMOLOGIES} PT tables, found {len(saved_files)}."
    )

validation_records = []
for file_path in saved_files:
    with np.load(file_path, allow_pickle=False) as saved:
        table = saved["pt_table"]
        k_saved = saved["k_native"]
        ells_saved = saved["ells"]
        labels_saved = saved["component_labels"]
        metadata = json.loads(str(saved["metadata_json"].item()))
        cosmo_saved = json.loads(str(saved["cosmology_json"].item()))

        assert table.shape == (
            EXPECTED_N_ELL, EXPECTED_N_K_NATIVE, EXPECTED_N_COMPONENTS
        )
        assert np.isfinite(table).all()
        assert np.all(np.diff(k_saved) > 0)
        assert np.array_equal(ells_saved, np.asarray(ELL_FID, dtype=int))
        assert labels_saved.tolist() == COMPONENT_LABELS
        assert metadata["cosmology_index"] == int(file_path.stem.split("_")[-1])

        idx = metadata["cosmology_index"]
        design_row = cosmology_design.loc[
            cosmology_design["cosmology_index"] == idx
        ].iloc[0]
        for name in COSMOLOGY_PARAMETERS:
            assert np.isclose(cosmo_saved[name], design_row[name], rtol=0, atol=1e-13)

        assert np.allclose(k_saved, native_k_reference, rtol=0, atol=1e-12)

        validation_records.append({
            "cosmology_index": idx,
            "file_size_MB": file_path.stat().st_size / 1e6,
            "table_rms": float(np.sqrt(np.mean(table**2))),
            "finite": bool(np.isfinite(table).all()),
        })

validation_df = pd.DataFrame(validation_records).sort_values("cosmology_index")
display(validation_df.describe(include="all"))
assert validation_df["cosmology_index"].nunique() == N_COSMOLOGIES
print("All saved PT tables passed reload and integrity checks.")

## 8. Inspect the cosmological design and component variation

The first plot shows the sampled cosmologies in each coordinate. The second gives a quick view of how the RMS amplitude of the 19 PT components varies across the design. These are diagnostics only; no emulator model is fitted in this notebook.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, name in zip(axes.flat, COSMOLOGY_PARAMETERS):
    ax.hist(cosmology_design[name], bins=18)
    ax.set_xlabel(name)
    ax.set_ylabel("Count")
    ax.grid(alpha=0.2)
fig.suptitle("300-point revised unanchored cosmology LHS")
fig.tight_layout()
plt.show()

# Summarize component RMS across cosmologies, averaging over ell and native k.
component_rms = np.zeros((N_COSMOLOGIES, EXPECTED_N_COMPONENTS))
for file_path in saved_files:
    idx = int(file_path.stem.split("_")[-1])
    with np.load(file_path, allow_pickle=False) as saved:
        table = saved["pt_table"]
    component_rms[idx] = np.sqrt(np.mean(table**2, axis=(0, 1)))

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(
    np.log10(np.maximum(component_rms, 1e-30)).T,
    aspect="auto",
    interpolation="nearest",
)
ax.set_yticks(np.arange(EXPECTED_N_COMPONENTS))
ax.set_yticklabels(COMPONENT_LABELS)
ax.set_xlabel("Cosmology index")
ax.set_title("log10 RMS of each PT component across the cosmology design")
fig.colorbar(im, ax=ax, label="log10 RMS")
fig.tight_layout()
plt.show()

## 9. Dataset contents and handoff

The run directory is:

`./training_data/velocileptors_pt_lhs_v2_restricted/`

- `pt_tables/cosmo_0000.npz` through `cosmo_0299.npz`: one native PT table per cosmology.
- `cosmology_design.csv`: exact 300-point 8D input design.
- `progress.csv`: per-cosmology completion and basic diagnostics.
- `run_config.json`: parameter bounds, seed, theory settings, table schema, and component ordering.

The expected complete dataset contains 300 files, each with a `(3, 117, 19)` PT table and its native $k$ grid.

**Next step:** use the independently validated physical-to-standard nuisance mapping and 19-component contraction to reconstruct spectra from these saved tables. Only after that reconstruction is accurate across multiple cosmologies and nuisance settings should the emulator training notebook consume this dataset.

This notebook does not generate the 52-anchor comparison set, fit an emulator, or interpolate the native PT tables onto a different $k$ grid.